# 🎯 RFM Segmentation — Recency, Frequency, Monetary

**Phase 4** of the Customer Intelligence Platform.

This notebook builds per-customer RFM features from cleaned transactions, assigns quantile scores (1–5), and clusters customers into five actionable segments using K-Means:

| Segment | Profile |
|---------|---------|
| **Champions** | Bought recently, often, and spent the most |
| **Loyal** | Frequent buyers with good spend |
| **Potential** | Recent customers with low-to-moderate frequency |
| **At-Risk** | Former good customers who haven't purchased recently |
| **Lost** | Inactive, low-frequency, low-spend |

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import preprocess
from src.rfm import build_rfm_table, score_rfm, segment_customers, SEGMENT_LABELS
from src.visualizations import set_plot_style, save_fig

set_plot_style()
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("✅ Setup complete")

## 1. Load & clean transactions

We use ``preprocess()`` which loads raw data, removes returns/cancellations and anonymous customers, and engineers features.

In [ ]:
df = preprocess(keep_returns=False, save=False)
print(f"\nCleaned transactions: {len(df):,} rows")
df.head()

## 2. Build the RFM table

One row per customer: **Recency** (days since last purchase), **Frequency** (unique invoices), **Monetary** (total spend).

In [ ]:
rfm = build_rfm_table(df)
rfm.describe()

In [ ]:
# Pairwise distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.histplot(rfm[col], bins=40, kde=True, ax=ax)
    ax.set_title(col)
fig.suptitle("RFM feature distributions", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "rfm_distributions", subdir="rfm")
plt.show()

## 3. Quantile scoring (1–5)

Each RFM dimension is binned into 5 quantile-based scores.

In [ ]:
rfm = score_rfm(rfm)
rfm[["Recency", "R_Score", "Frequency", "F_Score", "Monetary", "M_Score", "RFM_Score"]].head(10)

In [ ]:
# Score distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col, color in zip(axes, ["R_Score", "F_Score", "M_Score"], ["#4C72B0", "#55A868", "#C44E52"]):
    sns.countplot(x=rfm[col], ax=ax, color=color)
    ax.set_title(f"{col} distribution")
    ax.set_xlabel("Score (1–5)")
fig.suptitle("RFM score distributions", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "rfm_score_distributions", subdir="rfm")
plt.show()

## 4. K-Means clustering → 5 segments

In [ ]:
rfm = segment_customers(rfm, n_clusters=5)
rfm.head()

### Segment size & average RFM

In [ ]:
segment_summary = rfm.groupby("Segment").agg(
    customers=("Recency", "count"),
    avg_recency=("Recency", "mean"),
    avg_frequency=("Frequency", "mean"),
    avg_monetary=("Monetary", "mean"),
).reindex(SEGMENT_LABELS)
segment_summary["%"] = (segment_summary["customers"] / segment_summary["customers"].sum() * 100).round(1)
segment_summary

In [ ]:
# Segment sizes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie
axes[0].pie(
    segment_summary["customers"],
    labels=segment_summary.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("Set2", len(segment_summary)),
    startangle=140,
)
axes[0].set_title("Segment size distribution")

# Avg RFM per segment
metrics = segment_summary[["avg_recency", "avg_frequency", "avg_monetary"]]
metrics.columns = ["Recency", "Frequency", "Monetary"]
sns.heatmap(metrics, annot=True, fmt=",.1f", cmap="YlGnBu_r", ax=axes[1])
axes[1].set_title("Average RFM per segment")

fig.tight_layout()
save_fig(fig, "rfm_segments_overview", subdir="rfm")
plt.show()

### RFM scatter (Recency × Monetary, coloured by segment)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for seg in SEGMENT_LABELS:
    mask = rfm["Segment"] == seg
    ax.scatter(rfm.loc[mask, "Recency"], rfm.loc[mask, "Monetary"],
               label=seg, alpha=0.5, s=20)
ax.set_xlabel("Recency (days)")
ax.set_ylabel("Monetary (£)")
ax.set_title("Customer segments — Recency vs Monetary")
ax.legend(title="Segment")
save_fig(fig, "rfm_scatter_recency_monetary", subdir="rfm")
plt.show()

### Top RFM scores per segment

In [ ]:
top_per_segment = (
    rfm.groupby("Segment")
    .apply(lambda g: g.nlargest(3, "Monetary")[["Recency", "Frequency", "Monetary", "RFM_Score"]])
    .reindex(SEGMENT_LABELS)
)
top_per_segment

## 5. Summary & next steps

### Key takeaways
- Customers are segmented into **5 actionable groups** based on pure transactional behaviour.
- **Champions** have the lowest recency and highest frequency/monetary — they are the core revenue drivers.
- **At-Risk** and **Lost** segments warrant targeted retention campaigns.
- RFM scores serve as features for the downstream **LTV** (Phase 5) and **Churn** (Phase 6) models.

### Next notebook
→ **`03_ltv.ipynb`** (Phase 5): predict Customer Lifetime Value using BG/NBD + Gamma-Gamma probabilistic models.

---
© 2026 AdamAI-Systems.